**Code :** [GitHub · Kukai-7/industrial-data-valuation](https://github.com/Kukai-7/industrial-data-valuation) · **Données :** [Hugging Face · synthetic-industrial-data-valuation](https://huggingface.co/datasets/Kukai-7/synthetic-industrial-data-valuation) · **Démo :** [Space](https://huggingface.co/spaces/Kukai-7/industrial-data-valuation-demo)

**Résumé (FR).** Ce notebook fabrique trois tables industrielles synthétiques (capteurs, maintenance, approvisionnement) reliées par des clés cohérentes, pour illustrer les méthodes de valorisation de la donnée comme actif incorporel. Limites d'abord : aucune donnée réelle, des paramètres plausibles mais inventés, et une illustration finale qui donne un classement relatif, pas un prix.

**Abstract (EN).** This notebook generates three synthetic industrial tables (sensors, maintenance, supply chain) linked by consistent keys, to illustrate valuation methods for data as an intangible asset. Limits first: no real data, plausible but invented parameters, and a closing illustration that gives a relative ranking, not a price.

# Génération des données synthétiques

> *Synthetic data generated for methodological illustration. No real industrial data.*

Trois tables, une graine fixe, un export Parquet dans `data/synthetic/`.

| Table | Grain | Clés |
|---|---|---|
| `sensors` | un relevé toutes les 6 heures, 40 équipements × 3 grandeurs, 180 jours | `reading_id`, `asset_id`, `sensor_id` |
| `maintenance` | un ordre de travail | `work_order_id`, `asset_id` → `sensors`, `purchase_order_id` → `supply_chain` |
| `supply_chain` | un bon de commande de pièces | `purchase_order_id`, `supplier_id` |

Les liens entre tables sont voulus : la vibration d'un équipement monte dans les trois jours qui précèdent une panne, chaque pièce utilisée vient d'un bon de commande déjà livré et de la même famille, et chaque fournisseur a son propre taux de défaut.

In [ ]:
# Synthetic data generated for methodological illustration. No real industrial data.
import numpy as np
import pandas as pd

SEED = 2026
rng = np.random.default_rng(SEED)

N_ASSETS, N_SUPPLIERS = 40, 12
DEBUT = pd.Timestamp("2025-01-01")
JOURS = 180
ASSETS = [f"AS-{i:03d}" for i in range(1, N_ASSETS + 1)]
SUPPLIERS = [f"SUP-{i:02d}" for i in range(1, N_SUPPLIERS + 1)]
FAMILLES = ["bearing", "seal", "motor", "valve", "sensor", "controller"]

## 1 · Approvisionnement

In [ ]:
# ---------------------------------------------------------------- supply_chain : bons de commande de pièces
fiab = rng.beta(2, 18, N_SUPPLIERS)                       # taux de défaut propre à chaque fournisseur
delai = rng.integers(7, 46, N_SUPPLIERS)                  # délai promis propre à chaque fournisseur (jours)
prix_base = dict(zip(FAMILLES, [180, 45, 1200, 320, 260, 900]))
n_po = 900
sup = rng.integers(0, N_SUPPLIERS, n_po)
fam = rng.choice(FAMILLES, n_po)
order = DEBUT + pd.to_timedelta(rng.integers(0, JOURS, n_po), unit="D")
promis = order + pd.to_timedelta(delai[sup], unit="D")
retard = np.maximum(0, rng.normal(2, 6, n_po)).round()
supply_chain = pd.DataFrame({
    "purchase_order_id": [f"PO-{i:05d}" for i in range(1, n_po + 1)],
    "supplier_id": np.array(SUPPLIERS)[sup],
    "part_family": fam,
    "order_date": order,
    "promised_date": promis,
    "delivered_date": promis + pd.to_timedelta(retard, unit="D"),
    "quantity": rng.integers(1, 40, n_po),
    "unit_price_eur": np.round([prix_base[f] * rng.lognormal(0, .18) for f in fam], 2),
    "defect_rate": np.round(np.clip(fiab[sup] + rng.normal(0, .01, n_po), 0, 1), 4),
})
supply_chain["lead_time_days"] = (supply_chain.delivered_date - supply_chain.order_date).dt.days

## 2 · Maintenance

In [ ]:
# ---------------------------------------------------------------- maintenance : ordres de travail
n_wo = 1200
asset = rng.integers(0, N_ASSETS, n_wo)
typ = rng.choice(["preventive", "corrective", "inspection"], n_wo, p=[.45, .35, .20])
ouvert = DEBUT + pd.to_timedelta(rng.integers(0, JOURS * 24, n_wo), unit="h")
arret = np.where(typ == "corrective", rng.gamma(2.2, 4.0, n_wo), np.where(typ == "preventive", rng.gamma(1.5, 1.5, n_wo), rng.gamma(1.0, .5, n_wo)))
fam_wo = rng.choice(FAMILLES, n_wo)
avec_piece = (typ != "inspection") & (rng.random(n_wo) < .7)
par_famille = {f: np.flatnonzero(fam == f) for f in FAMILLES}   # la pièce commandée est de la même famille que le composant
livre = (promis + pd.to_timedelta(retard, unit="D")).values
def choisir_po(f, t):                                        # une pièce déjà livrée à l'ouverture de l'ordre ; sinon, pas de pièce
    c = par_famille[f]; ok = c[livre[c] <= t.to_datetime64()]
    return rng.choice(ok) if len(ok) else -1
po_idx = np.array([choisir_po(f, t) for f, t in zip(fam_wo, ouvert)])
avec_piece = avec_piece & (po_idx >= 0)
po_idx = np.maximum(po_idx, 0)
maintenance = pd.DataFrame({
    "work_order_id": [f"WO-{i:05d}" for i in range(1, n_wo + 1)],
    "asset_id": np.array(ASSETS)[asset],
    "opened_at": ouvert,
    "closed_at": ouvert + pd.to_timedelta(np.round(arret * 60), unit="min"),
    "work_type": typ,
    "component_family": fam_wo,
    "downtime_hours": np.round(arret, 2),
    "labour_hours": np.round(arret * rng.uniform(.6, 1.4, n_wo) + 1, 2),
    "purchase_order_id": np.where(avec_piece, supply_chain.purchase_order_id.values[po_idx], None),
    "supplier_id": np.where(avec_piece, supply_chain.supplier_id.values[po_idx], None),
})
maintenance["parts_cost_eur"] = np.where(avec_piece, np.round(supply_chain.unit_price_eur.values[po_idx] * rng.integers(1, 5, n_wo), 2), 0.0)

## 3 · Capteurs

In [ ]:
# ---------------------------------------------------------------- sensors : relevés toutes les 6 heures
pas = pd.date_range(DEBUT, periods=JOURS * 4, freq="6h")
METRIQUES = {"temperature_c": (65, 4), "vibration_mm_s": (2.8, .5), "power_kw": (45, 6)}
corr = maintenance[maintenance.work_type == "corrective"]
blocs = []
for a_i, a in enumerate(ASSETS):
    derive = np.zeros(len(pas))
    for t in corr.opened_at[corr.asset_id == a]:            # la vibration monte dans les 3 jours qui précèdent une panne
        k = np.searchsorted(pas, t)
        derive[max(0, k - 12):k] += np.linspace(0, .9, min(12, k))
    niveau = rng.normal(0, .05, 3)
    for m_i, (m, (mu, sd)) in enumerate(METRIQUES.items()):
        v = mu * (1 + niveau[m_i]) + rng.normal(0, sd, len(pas))
        if m == "vibration_mm_s": v = v + derive
        manque = rng.random(len(pas)) < .02                 # 2 % de relevés manquants
        blocs.append(pd.DataFrame({"asset_id": a, "sensor_id": f"{a}-{m_i + 1}", "timestamp": pas, "metric": m,
                                   "value": np.where(manque, np.nan, np.round(v, 3)), "quality_flag": np.where(manque, "missing", "ok")}))
sensors = pd.concat(blocs, ignore_index=True)
sensors.insert(0, "reading_id", np.arange(1, len(sensors) + 1))

TABLES = {"sensors": sensors, "maintenance": maintenance, "supply_chain": supply_chain}
for t in TABLES.values():                                     # unité de temps explicite (stable entre versions de pandas)
    for c in t.columns:
        if str(t[c].dtype).startswith("datetime64"): t[c] = t[c].astype("datetime64[ms]")

## 4 · Contrôles de cohérence

In [ ]:
m = maintenance.dropna(subset=["purchase_order_id"]).merge(supply_chain, on="purchase_order_id", suffixes=("", "_po"))
controles = {
    "lignes par table": {k: len(v) for k, v in TABLES.items()},
    "clés primaires uniques": all(TABLES[t][k].is_unique for t, k in [("sensors", "reading_id"), ("maintenance", "work_order_id"), ("supply_chain", "purchase_order_id")]),
    "équipements de maintenance présents dans sensors": set(maintenance.asset_id) <= set(sensors.asset_id),
    "fournisseur identique à celui du bon de commande": bool((m.supplier_id == m.supplier_id_po).all()),
    "pièce de la même famille que le composant": bool((m.component_family == m.part_family).all()),
    "part de relevés manquants": round(float(sensors.value.isna().mean()), 4),
}
controles

## 5 · Export Parquet

In [ ]:
import os
os.makedirs("../data/synthetic", exist_ok=True)
for nom, t in TABLES.items():
    t.to_parquet(f"../data/synthetic/{nom}.parquet", index=False)
    print(f"data/synthetic/{nom}.parquet", t.shape)

## 6 · Illustration : les trois approches sur ces tables

**Limites avant résultats.** Toutes les hypothèses économiques ci-dessous (coûts de reconstitution, prix de référence, coût d'une heure d'arrêt) sont **inventées pour l'exemple**. Le résultat est un classement relatif entre les trois tables, pas un prix opposable.

- **Coût :** ce qu'il faudrait dépenser pour reconstituer la table (capteurs posés, part d'outil logiciel, temps de saisie et de nettoyage).
- **Marché :** volume × prix de référence par enregistrement, moins une décote de 50 % faute de comparables.
- **Revenus :** gains annuels que la table rend possibles, actualisés sur 5 ans à 12 % ; ils sont mesurés sur les données elles-mêmes (pannes précédées d'une alerte de vibration, écart de défaut entre fournisseurs).

Les trois montants sont mélangés selon leur confiance (comme dans le notebook 01) pour donner $Val$, puis passés dans la formule de référence $V = QRU \times \sum (Val \times ICF) / V_p$ : $QRU$ est ici la complétude de la table, $ICF = 1$ (cession unique), $V_p = 1{,}05$ (traitement léger), et la somme porte sur les trois tables du portefeuille.

In [ ]:
H = dict(infra={"sensors": 180_000, "maintenance": 30_000, "supply_chain": 20_000},   # € : capteurs posés, part d'outil de GMAO / d'ERP (hypothèse)
         heures_enreg={"sensors": 0, "maintenance": .25, "supply_chain": .2},            # heures de saisie et de nettoyage par enregistrement (hypothèse)
         cout_heure=45, prix_ref={"sensors": .01, "maintenance": 2.0, "supply_chain": 1.0},  # €/heure, €/enregistrement (hypothèses)
         decote=.5, cout_heure_arret=500, capture=.3, horizon=5, taux=.12,
         poids={"cout": .30, "marche": .25, "revenus": .45}, confiance={"cout": .9, "marche": .5, "revenus": .7},
         icf=1.0, vp=1.05)
annuite = (1 - (1 + H["taux"]) ** -H["horizon"]) / H["taux"]
an = 365 / 180                                                                         # mise à l'année de la fenêtre de 180 jours

# Capteurs : part des pannes précédées d'une vibration anormale (seuil : moyenne + 2 écarts-types) et heures d'arrêt évitables
vib = sensors[sensors.metric == "vibration_mm_s"].set_index(["asset_id", "timestamp"]).value.sort_index()
seuil = vib.mean() + 2 * vib.std()
pannes = maintenance[maintenance.work_type == "corrective"]
detectee = [bool((vib.loc[a].loc[t - pd.Timedelta("3D"):t] > seuil).any()) for a, t in zip(pannes.asset_id, pannes.opened_at)]
gain_capteurs = pannes.downtime_hours[detectee].sum() * H["cout_heure_arret"] * H["capture"] * an
# Maintenance : planifier une partie des pannes ramène leur durée à celle d'un préventif
ecart = pannes.downtime_hours.mean() - maintenance[maintenance.work_type == "preventive"].downtime_hours.mean()
gain_maint = ecart * len(pannes) * H["cout_heure_arret"] * H["capture"] * .5 * an
# Approvisionnement : acheter auprès du tiers de fournisseurs le moins défaillant
achats = (supply_chain.quantity * supply_chain.unit_price_eur).sum()
moy = supply_chain.groupby("supplier_id").defect_rate.mean()
gain_appro = achats * (supply_chain.defect_rate.mean() - moy.nsmallest(4).mean()) * an

gains = {"sensors": gain_capteurs, "maintenance": gain_maint, "supply_chain": gain_appro}
w = {k: H["poids"][k] * H["confiance"][k] for k in H["poids"]}
lignes = []
for nom, t in TABLES.items():
    a = dict(cout=H["infra"][nom] + len(t) * H["heures_enreg"][nom] * H["cout_heure"],
             marche=len(t) * H["prix_ref"][nom] * (1 - H["decote"]),
             revenus=gains[nom] * annuite)
    val = sum(w[k] * a[k] for k in a) / sum(w.values())                              # Val : mélange des trois approches pondéré par la confiance
    qru = 1 - t.isna().any(axis=1).mean() if nom == "sensors" else 1.0             # QRU : ici la complétude des enregistrements
    lignes.append(dict(table=nom, **a, Val=val, QRU=qru, V=qru * val * H["icf"] / H["vp"]))
V = pd.DataFrame(lignes).set_index("table")
V["rang"] = V.V.rank(ascending=False).astype(int)
print(f"Pannes précédées d'une alerte de vibration : {np.mean(detectee):.0%} (seuil {seuil:.2f} mm/s)")
print(f"Portefeuille (somme des trois tables) : {V.V.sum():,.0f} unités relatives".replace(",", " "))
V.round(2)

**Lecture.** La jambe revenus, mesurée sur les données elles-mêmes, fait l'essentiel de l'écart entre tables, comme dans le notebook 01. Changer une seule hypothèse inventée (le coût d'une heure d'arrêt, par exemple) peut inverser le classement : c'est la raison d'être de l'analyse de sensibilité du notebook 01.

## Reproducibility & License

- **Exécution :** *Run All* suffit ; graine `SEED = 2026`, aucune donnée externe.
- **Données :** *Synthetic data generated for methodological illustration. No real industrial data.*
- **Licence :** code sous licence MIT ; données sous CC BY 4.0.